# Green Patent Detection (PatentSBERTa): Active Learning + LLM→Human HITL

### Part A: Baseline Model (Frozen Embeddings)

In [2]:
# Installing and importing all necessary libraries

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModel

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, classification_report


1) Loading the 50k parquet and check columns

In [3]:
from datasets import load_dataset

ds = load_dataset("AI-Growth-Lab/patents_claims_1.5m_traim_test", split="train")
print(ds)


Dataset({
    features: ['id', 'date', 'text', 'A01B', 'A01C', 'A01D', 'A01F', 'A01G', 'A01H', 'A01J', 'A01K', 'A01L', 'A01M', 'A01N', 'A21B', 'A21C', 'A21D', 'A22B', 'A22C', 'A23B', 'A23C', 'A23D', 'A23F', 'A23G', 'A23J', 'A23K', 'A23L', 'A23N', 'A23P', 'A23V', 'A23Y', 'A24B', 'A24C', 'A24D', 'A24F', 'A41B', 'A41C', 'A41D', 'A41F', 'A41G', 'A41H', 'A42B', 'A42C', 'A43B', 'A43C', 'A43D', 'A44B', 'A44C', 'A44D', 'A45B', 'A45C', 'A45D', 'A45F', 'A46B', 'A46D', 'A47B', 'A47C', 'A47D', 'A47F', 'A47G', 'A47H', 'A47J', 'A47K', 'A47L', 'A61B', 'A61C', 'A61D', 'A61F', 'A61G', 'A61H', 'A61J', 'A61K', 'A61L', 'A61M', 'A61N', 'A61P', 'A61Q', 'A62B', 'A62C', 'A62D', 'A63B', 'A63C', 'A63D', 'A63F', 'A63G', 'A63H', 'A63J', 'A63K', 'B01B', 'B01D', 'B01F', 'B01J', 'B01L', 'B02B', 'B02C', 'B03B', 'B03C', 'B03D', 'B04B', 'B04C', 'B05B', 'B05C', 'B05D', 'B06B', 'B07B', 'B07C', 'B08B', 'B09B', 'B09C', 'B21B', 'B21C', 'B21D', 'B21F', 'B21G', 'B21H', 'B21J', 'B21K', 'B21L', 'B22C', 'B22D', 'B22F', 'B23B', '

2. Load train+test and put into one pandas df

In [4]:
df = ds.to_pandas()
print(df.shape)
df.head()


(1372910, 666)


,id,date,text,A01B,A01C,A01D,A01F,A01G,A01H,A01J,...,Y02B,Y02C,Y02D,Y02E,Y02P,Y02T,Y02W,Y04S,Y10S,Y10T
0,8788730,2014-07-22,1. A method for sending a keycode of a non-key...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,8621421,2013-12-31,1. A method executed at least in part in a com...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,9461433,2016-10-04,1. A light-emitting device comprising: a base;...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,9229528,2016-01-05,"1. An input apparatus, comprising: a plurality...",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8508147,2013-08-13,"1. A dimmer circuit, comprising: a bleeder as ...",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


3. Creating is_green_silver

In [6]:
#Finding any columns starting with Y02
y02_cols = [c for c in df.columns if str(c).startswith("Y02")]
print("Y02 columns:", y02_cols[:20], " ... total:", len(y02_cols))


Y02 columns: ['Y02A', 'Y02B', 'Y02C', 'Y02D', 'Y02E', 'Y02P', 'Y02T', 'Y02W']  ... total: 8


In [7]:
#CREATING THE LABEL
if len(y02_cols) > 0:
    df["is_green_silver"] = (df[y02_cols].sum(axis=1) > 0).astype(int)
else:
    raise ValueError("No Y02* columns found. We need to inspect how green tech is encoded in this dataset.")


In [8]:
#Checking balance
print(df["is_green_silver"].value_counts())


is_green_silver
0    1269218
1     103692
Name: count, dtype: int64


4. Creating the balanced 50k sample (25k green + 25k not green)

In [9]:
SEED = 42

green = df[df["is_green_silver"] == 1].sample(25000, random_state=SEED)
nong  = df[df["is_green_silver"] == 0].sample(25000, random_state=SEED)

balanced = pd.concat([green, nong], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

balanced["is_green_silver"].value_counts()


is_green_silver
0    25000
1    25000
Name: count, dtype: int64

5. Creating splits and save patents_50k_green.parquet

In [14]:
train = balanced.sample(frac=0.60, random_state=SEED)
rest = balanced.drop(train.index)

eval_ = rest.sample(frac=0.50, random_state=SEED)   # half of remaining = 20%
pool  = rest.drop(eval_.index)                      # remaining = 20%

train = train.copy(); eval_ = eval_.copy(); pool = pool.copy()
train["split"] = "train_silver"
eval_["split"]  = "eval_silver"
pool["split"]   = "pool_unlabeled"

final_df = pd.concat([train, pool, eval_], ignore_index=True)

final_df.to_parquet("patents_50k_green.parquet", index=False)
final_df["split"].value_counts()

final_df[final_df["split"] == "train_silver"] \
    .to_parquet("train_silver.parquet", index=False)

final_df[final_df["split"] == "eval_silver"] \
    .to_parquet("eval_silver.parquet", index=False)

final_df[final_df["split"] == "pool_unlabeled"] \
    .to_parquet("pool_unlabeled.parquet", index=False)

In [11]:
# From this point on, always load from the working dataset
final_df = pd.read_parquet("patents_50k_green.parquet")
print("Reloaded working dataset:", final_df.shape)

Reloaded working dataset: (50000, 668)


We moved to AI LAB to run the embeddings and logistic regression along with evaluation. Please refer to our python scripts in the repository.